# Notebook 5: Calibrate Router Handoff Thresholds

This notebook calibrates router crop/part handoff behavior from an offline eval set. It runs the maintained script surfaces instead of duplicating calibration logic in notebook cells.

Expected full-router eval layout:

```text
data/router_eval/
  id/<crop>/<part>/*
  negatives/off_crop/<label>/*
  negatives/non_plant/<label>/*
  ambiguous/<label>/*
  wrong_part/<crop>/<unsupported_part>/*
```

The calibration output recommends config overrides for router thresholds and evidence weights. It does not edit `config/base.json` automatically.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
    Path('/content/workspace'),
    Path('/content/project'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repo bootstrap basarisiz oldu. Mevcut bir checkout icin AADS_REPO_ROOT ayarlayin veya '
        'private GitHub auto-clone icin GH_TOKEN/GITHUB_TOKEN saglayin.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import install_colab_requirements, resolve_repo_root, running_in_colab

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

print(ROOT)

In [ ]:
from src.core.config_manager import get_config
from scripts.colab_repo_bootstrap import collect_notebook_access_report, print_notebook_access_report

CONFIG_FOR_ACCESS = get_config(environment='colab')
ROUTER_VLM_CFG = dict(dict(CONFIG_FOR_ACCESS.get('router', {})).get('vlm', {}))
ROUTER_MODEL_IDS = [
    str(model_id).strip()
    for model_id in list(dict(ROUTER_VLM_CFG.get('model_ids', {})).values())
    if str(model_id).strip()
]
ACCESS_REPORT = collect_notebook_access_report(repo_root=ROOT, hf_model_ids=ROUTER_MODEL_IDS)
print_notebook_access_report(ACCESS_REPORT, print_fn=print)


In [ ]:
import json
from pathlib import Path

from scripts.calibrate_router_surface import calibrate_router_surface
from scripts.evaluate_router_surface import discover_eval_samples, evaluate_router_surface

CONFIG_ENV = 'colab'
DEVICE = 'cuda'

ROUTER_EVAL_ROOT = 'data/router_eval'
BASELINE_EVAL_OUTPUT = '.runtime_tmp/router_eval.json'
CALIBRATION_OUTPUT = '.runtime_tmp/router_calibration.json'

RUN_BASELINE_EVAL = True
RUN_CALIBRATION = True

CALIBRATION_PRESET = 'quick'  # none, handoff, quick, docs
CUSTOM_SWEEPS = []
# Examples:
# CUSTOM_SWEEPS = ['router_min_confidence=0.55,0.65,0.75', 'router_min_margin=0.0,0.1,0.15']

MAX_VARIANTS = 128
TARGET_NEGATIVE_FALSE_ACCEPT_RATE = 0.05
MAX_CROP_ACCURACY_DROP = 0.02
MAX_PART_PRECISION_DROP = 0.02
INCLUDE_SAMPLES_IN_OUTPUT = False

def _write_json(path_value, payload):
    path = Path(path_value)
    if str(path):
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        print(f'[OUTPUT] wrote {path}')

root = Path(ROUTER_EVAL_ROOT)
print(f'[CONFIG] eval_root={root}')
print(f'[CONFIG] config_env={CONFIG_ENV} device={DEVICE} preset={CALIBRATION_PRESET}')


In [ ]:
eval_root = Path(ROUTER_EVAL_ROOT)
samples = discover_eval_samples(eval_root) if eval_root.exists() else []
print(f'[DATA] root={eval_root} exists={eval_root.exists()} samples={len(samples)}')

if not samples:
    print('[DATA] No router eval images found. Create the data/router_eval layout before running calibration.')
else:
    counts = {}
    for sample in samples:
        group = str(sample.get('group', 'unknown'))
        counts[group] = counts.get(group, 0) + 1
    for group, count in sorted(counts.items()):
        print(f'[DATA] {group}: {count}')


In [ ]:
if RUN_BASELINE_EVAL:
    baseline_result = evaluate_router_surface(
        Path(ROUTER_EVAL_ROOT),
        config_env=CONFIG_ENV,
        device=DEVICE,
    )
    _write_json(BASELINE_EVAL_OUTPUT, baseline_result)
    metrics = baseline_result.get('metrics', {})
    print('[BASELINE] metrics')
    for key in (
        'sample_count',
        'crop_accuracy',
        'negative_false_accept_rate',
        'abstention_rate',
        'part_non_unknown_precision',
        'part_recall',
        'wrong_part_rejection_rate',
        'mean_latency_ms',
        'p95_latency_ms',
    ):
        print(f'  {key}: {metrics.get(key)}')
else:
    print('[BASELINE] skipped')


In [ ]:
if RUN_CALIBRATION:
    calibration_result = calibrate_router_surface(
        Path(ROUTER_EVAL_ROOT),
        config_env=CONFIG_ENV,
        device=DEVICE,
        preset=CALIBRATION_PRESET,
        sweep_specs=CUSTOM_SWEEPS,
        include_current=True,
        max_variants=MAX_VARIANTS,
        target_negative_false_accept_rate=TARGET_NEGATIVE_FALSE_ACCEPT_RATE,
        max_crop_accuracy_drop=MAX_CROP_ACCURACY_DROP,
        max_part_precision_drop=MAX_PART_PRECISION_DROP,
        include_samples=INCLUDE_SAMPLES_IN_OUTPUT,
    )
    _write_json(CALIBRATION_OUTPUT, calibration_result)

    recommended = calibration_result.get('recommended', {})
    print('[CALIBRATION] recommended variant')
    print(json.dumps({
        'variant_id': recommended.get('variant_id'),
        'eligible': recommended.get('eligible'),
        'eligibility_reasons': recommended.get('eligibility_reasons'),
        'overrides': recommended.get('overrides'),
        'metrics': recommended.get('metrics'),
    }, indent=2))
else:
    print('[CALIBRATION] skipped')


In [ ]:
recommended = (globals().get('calibration_result') or {}).get('recommended', {})
overrides = dict(recommended.get('overrides') or {})
if overrides:
    print('[CONFIG PREVIEW] Apply these dotted-path values to the relevant config after review:')
    print(json.dumps(overrides, indent=2))
else:
    print('[CONFIG PREVIEW] No override recommendation is available yet.')
